<a href="https://colab.research.google.com/github/MouslimRaza/smart-finance-assistant/blob/main/Starter_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gradio pandas hands-on-ai --quiet

In [ ]:
import pandas as pd
import numpy as np
import os, io, warnings
warnings.filterwarnings('ignore')

os.environ['HANDS_ON_AI_SERVER']  = 'https://ollama.serveur.au'
os.environ['HANDS_ON_AI_MODEL']   = 'llama3.2'
os.environ['HANDS_ON_AI_API_KEY'] = 'isys2001-assignment-key'

from hands_on_ai.chat import get_response
print("✅ Setup complete!")

In [ ]:
step1 = """
STEP 1: UNDERSTAND THE PROBLEM
===============================
Many young people who enjoy gaming, fitness, and entertainment
have no clear picture of where their money goes each month.
Subscriptions, game purchases, gym fees, cinema outings, and
café visits add up quickly without feeling significant in the moment.

The Smart Finance Assistant solves this by:
  1. Accepting a CSV file of personal transactions
  2. Automatically cleaning and categorising the data
  3. Showing a clear spending breakdown by category
  4. Offering a chatbot (FinBot) that answers finance questions
  5. Retrieving relevant saving tips from a knowledge base (RAG)
  6. Calculating how long it takes to reach a savings goal

TARGET USER:
  A young person who games on Xbox, goes to the gym, watches
  movies at MCine, and enjoys tea at Maison de Thé.
  Someone who wants quick spending insights without manually
  going through bank statements.

FINANCE PROBLEM BEING SOLVED:
  Lack of spending awareness — small daily purchases like tea at
  Maison de Thé and impulse Xbox game buys feel harmless but
  accumulate into hundreds of dollars of untracked spending per month.
"""
print(step1)

In [ ]:
step2 = """
STEP 2: IDENTIFY INPUTS AND OUTPUTS
=====================================

COMPONENT 1 — CSV Analysis
  Input  : CSV file with columns: Date, Amount, Category, Description
            Amounts may include $ signs and negatives (refunds)
  Output : Spending totals per category, percentages, recommendations

  Example Input Row:
    2026-03-05, $69.99, Gaming, Xbox Game Pass Ultimate 3 Months

  Example Output:
    Gaming      $312.45  (38.2%)  ████████
    Gym         $145.00  (17.8%)  ███
    Cinema      $ 86.00  (10.5%)  ██
    Tea & Café  $ 72.50   (8.9%)  █

COMPONENT 2 — Chatbot (FinBot)
  Input  : User question (e.g. "Am I spending too much on Xbox games?")
           Optional: spending summary from CSV as context
  Output : Plain-English personalised finance advice from FinBot

COMPONENT 3 — RAG (Finance Tips)
  Input  : User question (e.g. "How do I cut my gaming spend?")
  Output : Answer retrieved from the financial tips knowledge base

COMPONENT 4 — Savings Calculator
  Input  : Monthly income ($)
            Monthly expenses ($)
            Savings goal amount ($)
  Output : Months to reach goal, weekly/daily savings, savings rate %

COMPONENT 5 — Gradio UI
  Input  : All of the above via a browser interface
  Output : Formatted results displayed in the browser app
"""
print(step2)

In [ ]:
step3 = """
STEP 3: WORK THE PROBLEM BY HAND
==================================

--- Example 1: Cleaning an Amount ---

Raw value from CSV  :  "$69.99"
Step 1 — Remove $   :  "69.99"
Step 2 — To float   :  69.99
Result              :  69.99  ✓

Raw value from CSV  :  "-$15.00"
Step 1 — Remove $   :  "-15.00"
Step 2 — To float   :  -15.00
Result              :  -15.00  (this is a refund)  ✓

--- Example 2: Spending by Category ---

Transactions:
  2026-03-05  $69.99   Gaming      Xbox Game Pass Ultimate
  2026-03-08  $60.00   Gym         Monthly Gym Membership
  2026-03-12  $17.00   Cinema      MCine Moka - 2 Tickets Avengers
  2026-03-14  $8.50    Tea & Café  Maison de Thé - Green Tea Moka
  2026-03-20  -$15.00  Refund      Xbox Store - Refund Cancelled DLC

Positive spending only (ignore refunds):
  Gaming     = $69.99
  Gym        = $60.00
  Cinema     = $17.00
  Tea & Café = $8.50
  TOTAL      = $155.49

Percentages:
  Gaming     = 69.99 / 155.49 * 100 = 45.0%
  Gym        = 60.00 / 155.49 * 100 = 38.6%
  Cinema     = 17.00 / 155.49 * 100 = 10.9%
  Tea & Café =  8.50 / 155.49 * 100 =  5.5%
  CHECK: 45.0 + 38.6 + 10.9 + 5.5 = 100.0%  ✓

Refunds total = $15.00
Net spent     = $155.49 - $15.00 = $140.49

--- Example 3: Savings Calculator ---

Monthly income   = $2,500
Monthly expenses = $1,800
Monthly savings  = $2,500 - $1,800 = $700

Savings goal = $5,000 (e.g. new Xbox Series X + accessories + games)
Months needed = $5,000 / $700 = 7.14 months ≈ 8 months

Weekly savings = $700 / 4.33 = $161.66 per week
Savings rate   = $700 / $2,500 * 100 = 28.0%  (above 20% ✓)
"""
print(step3)

In [ ]:
step4 = """
STEP 4: PSEUDOCODE
===================

--- load_and_clean(file) ---
  READ csv file into a DataFrame
  CHECK that columns Date, Amount, Category, Description exist
    IF missing → raise error with helpful message
  FOR each row in Amount column:
    REMOVE dollar signs, commas, spaces using regex
    CONVERT to float (set invalid values to NaN)
  DROP rows where Amount could not be converted
  FILL missing Category with "Uncategorised"
  RETURN cleaned DataFrame

--- analyse_spending(df) ---
  SPLIT df into:
    positive_spending = rows where Amount > 0
    refunds           = rows where Amount < 0
  total_spent   = SUM of positive_spending amounts
  total_refunds = ABS(SUM of refund amounts)
  net_spent     = total_spent - total_refunds
  GROUP positive_spending by Category:
    calculate: sum, average, count per category
  CALCULATE percentage = category_total / total_spent * 100
  SORT categories by total descending
  RETURN dict with all the above values

--- make_recommendations(analysis) ---
  FOR each category in analysis:
    IF Gaming AND total > $50:
      ADD tip about 48-hour wishlist rule for Xbox purchases
    IF Gym AND total > $80:
      ADD tip about off-peak memberships and block PT sessions
    IF Cinema AND total > $40:
      ADD tip about MCine loyalty card and off-peak days
    IF Tea & Café AND total > $30:
      ADD tip about brewing tea at home instead of Maison de Thé daily
  IF no tips triggered:
    ADD generic positive message
  RETURN all tips as a formatted string

--- savings_calculator(income, expenses, goal) ---
  VALIDATE: income > 0, expenses >= 0, goal > 0
  monthly_savings = income - expenses
  IF monthly_savings <= 0:
    RETURN error message showing the shortfall amount
  months_needed = goal / monthly_savings
  weekly_saving = monthly_savings / 4.33
  savings_rate  = monthly_savings / income * 100
  RETURN formatted report with all calculations

--- chat_finbot(user_message, spending_context) ---
  BUILD system prompt with FinBot personality (Xbox/gym/MCine/Maison de Thé)
  IF spending_context available:
    APPEND context to system prompt
  CALL get_response(user_message, system=system_prompt)
  RETURN response text

--- rag_lookup(question) ---
  TRY hands_on_ai.rag.get_answer(question, documents)
  IF rag module not available:
    BUILD prompt: "Answer ONLY from this document: [tips]"
    CALL get_response(prompt)
  RETURN answer

--- Gradio UI ---
  CREATE app with 4 tabs:
    Tab 1: File upload → load_and_clean + analyse_spending + full_report
    Tab 2: Chat input + optional CSV → chat_finbot with spending context
    Tab 3: Text question → rag_lookup answer
    Tab 4: Income, expenses, goal → savings_calculator result
  LAUNCH app with share=True for public Colab link
"""
print(step4)